# 43 · Analysis: what it takes to get in, what a beginner earns, how hard it is to get hired

**Plan:** (nb 43). Reads: `dim_job` (BLS typical entry education / experience / training, O*NET job zone), `fct_job_attainment` (BLS Table 5.3, what workers actually hold), `fct_job_entry_signal` (the D5 range: BLS experience, ACS young-worker share, O*NET newcomer share), `fct_job_wage_nyc` (NYS DOL NYC entry / median / experienced wage), `fct_job_employment_wage_annual` + `fct_cpi_annual` (OEWS NY metro percentiles, real 2025 $), `fct_job_skill` and `fct_job_software` (O*NET), `fct_job_projection` 2022-32 for openings weights. Changes nothing.

**Questions**
1. What do degree-reachable jobs require: education, experience, training; and what do the people in them actually hold?
2. What does a beginner earn in NYC, by job and by featured area, and how has real pay moved since 2019?
3. How junior-friendly is each job: do the three D5 signals agree, and which large jobs are open to newcomers?
4. Which knowledge, skills and software matter most across the jobs with the most openings?

**Rules:** universe = degree-reachable detailed jobs (`is_degree_entry`); all averages are weighted by 2022-32 openings unless a table says otherwise; entry wage = NYS DOL "entry" (mean of the bottom third of workers, not new graduates); real dollars = NY metro CPI, 2025 base.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from explore import q

pd.set_option("display.max_columns", 40, "display.max_colwidth", 52, "display.width", 240,
              "display.float_format", "{:,.1f}".format)


def md(s):
    display(Markdown(s))


def show(df, title=None):
    if title:
        md(f"**{title}**")
    display(df)


def wavg(s, w):
    m = s.notna() & w.notna()
    return (s[m] * w[m]).sum() / w[m].sum() if w[m].sum() else np.nan


jobs = q("""
select soc_code, soc_title, soc_level, major_code, entry_education, entry_education_rank, entry_experience, entry_training,
       is_degree_entry, is_all_other, job_zone, in_tech, in_health, in_business, in_education, ai_band
from dim_job
""")
proj = q("select soc_code, annual_openings_avg as openings, pct_change, quality_flag from fct_job_projection where edition = '2022-2032' and is_leaf")
J = jobs.merge(proj, on="soc_code", how="left")
U = J[J.is_degree_entry & (J.soc_level == "detailed")].copy()
U["openings"] = U.openings.fillna(0)
AREAS = [("Tech", "in_tech"), ("Healthcare", "in_health"), ("Business", "in_business"), ("Education", "in_education")]
print(f"degree-reachable detailed jobs: {len(U)}; with 2022-32 openings: {int((U.openings > 0).sum())}; openings {U.openings.sum():,.0f} a year")

degree-reachable detailed jobs: 221; with 2022-32 openings: 182; openings 133,684 a year


## 1. What it takes: BLS entry requirements, weighted by openings

In [2]:
def req_table(col):
    t = U.groupby(col, dropna=False).agg(jobs=("soc_code", "size"), openings=("openings", "sum"))
    t["share of openings %"] = t.openings / t.openings.sum() * 100
    return t.sort_values("openings", ascending=False).round(1)


show(req_table("entry_education"), "Typical education to enter")
show(req_table("entry_experience"), "Work experience in a related job")
show(req_table("entry_training"), "On-the-job training")
show(req_table("job_zone"), "O*NET job zone (1 little preparation … 5 extensive)")

**Typical education to enter**

,jobs,openings,share of openings %
entry_education,,,
Bachelor's degree,173,"124,876.0",93.4
Associate's degree,48,"8,807.7",6.6


**Work experience in a related job**

,jobs,openings,share of openings %
entry_experience,,,
None,174,"90,999.6",68.1
Less than 5 years,30,"22,403.2",16.8
5 years or more,17,"20,281.0",15.2


**On-the-job training**

,jobs,openings,share of openings %
entry_training,,,
None,180,"118,357.8",88.5
Moderate-term on-the-job training,26,"9,982.0",7.5
Long-term on-the-job training,8,"3,782.0",2.8
Internship/residency,4,"1,056.0",0.8
Short-term on-the-job training,3,506.0,0.4


**O*NET job zone (1 little preparation … 5 extensive)**

,jobs,openings,share of openings %
job_zone,,,
4.0,143,"116,623.6",87.2
3.0,52,"11,576.2",8.7
5.0,14,"3,537.0",2.6
NaN,10,"1,946.0",1.5
2.0,2,1.0,0.0


### What workers actually hold (BLS Table 5.3)
The entry level is one thing; the mix of degrees among people doing the job is another. `acs_inexact` marks profiles BLS borrowed from a broader Census occupation.

In [3]:
att = q("select soc_code, attainment_level, attainment_rank, pct_of_workers, acs_inexact from fct_job_attainment")
print(f"fct_job_attainment rows: {len(att):,}")
A = att.pivot_table(index="soc_code", columns="attainment_rank", values="pct_of_workers")
A.columns = ["<HS", "HS", "some college", "associate's", "bachelor's", "master's", "doctoral/prof"]
A["bachelor's+"] = A[["bachelor's", "master's", "doctoral/prof"]].sum(axis=1)
A["sub-bachelor's"] = A[["<HS", "HS", "some college", "associate's"]].sum(axis=1)
UA = U.merge(A, left_on="soc_code", right_index=True, how="left").merge(att.groupby("soc_code").acs_inexact.first().rename("borrowed_profile"), left_on="soc_code", right_index=True, how="left")
mix = UA.groupby("entry_education").apply(lambda d: pd.Series({c: wavg(d[c], d.openings) for c in ["sub-bachelor's", "bachelor's", "master's", "doctoral/prof", "bachelor's+"]}), include_groups=False)
show(mix.round(1), "Degree mix of workers (% of workers, openings-weighted) by the job's typical entry education")
creep = UA[(UA.entry_education == "Associate's degree")].nlargest(10, "openings")[["soc_code", "soc_title", "openings", "sub-bachelor's", "bachelor's+", "borrowed_profile"]]
show(creep.round(1), "Associate's-entry jobs with the most openings: how many workers hold a bachelor's or more")
show(UA[(UA.entry_education == "Bachelor's degree") & (UA["sub-bachelor's"] >= 40)].nlargest(12, "openings")[["soc_code", "soc_title", "openings", "sub-bachelor's", "bachelor's+", "borrowed_profile"]].round(1),
     "Bachelor's-entry jobs where 40%+ of workers have less than a bachelor's (open to associate's holders in practice)")
print(f"borrowed (acs_inexact) profiles among degree-reachable jobs: {int(UA.borrowed_profile.fillna(False).sum())} of {len(UA)}")

fct_job_attainment rows: 5,824


**Degree mix of workers (% of workers, openings-weighted) by the job's typical entry education**

,sub-bachelor's,bachelor's,master's,doctoral/prof,bachelor's+
entry_education,,,,,
Associate's degree,54.8,33.1,9.7,2.3,45.1
Bachelor's degree,27.7,46.1,22.5,3.7,72.3


**Associate's-entry jobs with the most openings: how many workers hold a bachelor's or more**

,soc_code,soc_title,openings,sub-bachelor's,bachelor's+,borrowed_profile
350,23-2011,Paralegals and Legal Assistants,"2,036.0",50.1,49.8,False
406,25-2011,"Preschool Teachers, Except Special Education","1,420.0",46.2,53.8,True
590,29-2034,Radiologic Technologists and Technicians,607.0,69.9,30.1,False
216,17-3011,Architectural and Civil Drafters,539.0,65.9,34.0,False
143,15-1231,Computer Network Support Specialists,459.0,50.1,49.8,True
903,43-4161,"Human Resources Assistants, Except Payroll and T...",429.0,48.9,51.0,False
306,19-4099,"Life, Physical, and Social Science Technicians, ...",394.0,48.4,51.5,True
508,27-4012,Broadcast Technicians,338.0,55.7,44.3,True
580,29-1292,Dental Hygienists,320.0,60.2,39.8,False
588,29-2032,Diagnostic Medical Sonographers,273.0,50.8,49.3,False


**Bachelor's-entry jobs where 40%+ of workers have less than a bachelor's (open to associate's holders in practice)**

,soc_code,soc_title,openings,sub-bachelor's,bachelor's+,borrowed_profile
6,11-1021,General and Operations Managers,"9,781.0",53.1,47.0,False
109,13-1199,"Business Operations Specialists, All Other","2,023.8",43.3,56.7,False
105,13-1151,Training and Development Specialists,"1,458.0",46.5,53.4,False
451,25-9044,"Teaching Assistants, Postsecondary","1,227.0",65.5,34.3,True
459,27-1011,Art Directors,932.0,41.7,58.2,True
73,11-9199,"Managers, All Other",858.8,41.6,58.4,True
149,15-1244,Network and Computer Systems Administrators,772.0,47.2,52.8,False
42,11-9021,Construction Managers,738.0,65.3,34.7,False
20,11-3012,Administrative Services Managers,689.7,42.6,57.5,False
129,13-2072,Loan Officers,665.0,45.9,54.0,True


borrowed (acs_inexact) profiles among degree-reachable jobs: 113 of 221


## 2. Starting pay: NYS DOL NYC entry wage
`entry_wage` = mean of the lower third of the wage distribution for the job in NYC (survey May 2025, aged to 2026 Q1). It is a floor-of-the-market number for all workers, not a new-graduate salary. `entry_wage_at_floor` marks the 22 jobs where it equals the minimum-wage floor.

In [4]:
wage = q("select soc_code, employment as nyc_employment, entry_wage, median_wage, experienced_wage, entry_wage_at_floor, median_at_floor from fct_job_wage_nyc")
print(f"fct_job_wage_nyc rows: {len(wage)}")
UW = U.merge(wage, on="soc_code", how="left")
print(f"degree-reachable jobs with an NYC entry wage: {int(UW.entry_wage.notna().sum())} of {len(UW)}; at the wage floor: {int(UW.entry_wage_at_floor.fillna(False).sum())}")
dist = UW.entry_wage.describe(percentiles=[.1, .25, .5, .75, .9]).round(0)
show(dist.to_frame("entry wage, degree-reachable jobs (unweighted)").T, "Distribution")
by_ed = UW.groupby("entry_education").apply(lambda d: pd.Series({
    "jobs": len(d), "entry wage (openings-weighted)": wavg(d.entry_wage, d.openings), "median wage (openings-weighted)": wavg(d.median_wage, d.openings),
    "entry wage, median job": d.entry_wage.median()}), include_groups=False)
show(by_ed.round(0), "By typical entry education")
rows = []
for name, col in AREAS:
    d = UW[UW[col].fillna(False).astype(bool)]
    rows.append([name, len(d), d.openings.sum(), wavg(d.entry_wage, d.openings), wavg(d.median_wage, d.openings), d.entry_wage.min(), d.entry_wage.max()])
show(pd.DataFrame(rows, columns=["area", "jobs", "openings", "entry wage (weighted)", "median wage (weighted)", "lowest entry", "highest entry"]).round(0), "By featured area (S7)")
cols = ["soc_code", "soc_title", "entry_education", "openings", "entry_wage", "median_wage", "experienced_wage"]
show(UW[UW.openings >= 500].nlargest(15, "entry_wage")[cols].round(0), "Highest entry wage among jobs with 500+ openings a year")
show(UW[UW.openings >= 500].nsmallest(15, "entry_wage")[cols].round(0), "Lowest entry wage among jobs with 500+ openings a year")
UW["entry_to_median"] = UW.entry_wage / UW.median_wage
print(f"entry ÷ median wage, degree-reachable jobs: median {UW.entry_to_median.median():.2f}, "
      f"openings-weighted {wavg(UW.entry_to_median, UW.openings):.2f} (a long climb where the ratio is low)")

fct_job_wage_nyc rows: 702
degree-reachable jobs with an NYC entry wage: 204 of 221; at the wage floor: 0


**Distribution**

,count,mean,std,min,10%,25%,50%,75%,90%,max
"entry wage, degree-reachable jobs (unweighted)",204.0,"77,022.0","26,914.0","35,546.0","49,373.0","58,056.0","69,556.0","89,968.0","115,006.0","182,986.0"


**By typical entry education**

,jobs,entry wage (openings-weighted),median wage (openings-weighted),"entry wage, median job"
entry_education,,,,
Associate's degree,48.0,"59,876.0","80,085.0","57,294.0"
Bachelor's degree,173.0,"84,747.0","129,838.0","71,530.0"


...

**Highest entry wage among jobs with 500+ openings a year**

,soc_code,soc_title,entry_education,openings,entry_wage,median_wage,experienced_wage
9,11-3021,Computer and Information Systems Managers,Bachelor's degree,"1,848.0","164,322.0","224,388.0","282,258.0"
10,11-3031,Financial Managers,Bachelor's degree,"3,334.0","161,264.0","242,789.0","327,546.0"
4,11-2022,Sales Managers,Bachelor's degree,"1,584.0","148,600.0","231,387.0","322,019.0"
219,53-2011,"Airline Pilots, Copilots, and Flight Engineers",Bachelor's degree,753.0,"144,502.0","260,830.0","387,776.0"
3,11-2021,Marketing Managers,Bachelor's degree,"1,857.0","133,848.0","205,522.0","255,266.0"
14,11-3121,Human Resources Managers,Bachelor's degree,681.0,"127,224.0","198,037.0","256,451.0"
61,15-1252,Software Developers,Bachelor's degree,"4,308.0","121,283.0","174,663.0","210,540.0"
26,11-9199,"Managers, All Other",Bachelor's degree,859.0,"115,735.0","181,934.0","240,991.0"
21,11-9111,Medical and Health Services Managers,Bachelor's degree,"1,738.0","115,316.0","170,884.0","237,011.0"
56,15-1241,Computer Network Architects,Bachelor's degree,514.0,"109,561.0","164,272.0","192,952.0"


**Lowest entry wage among jobs with 500+ openings a year**

,soc_code,soc_title,entry_education,openings,entry_wage,median_wage,experienced_wage
163,25-9044,"Teaching Assistants, Postsecondary",Bachelor's degree,"1,227.0","37,904.0","39,811.0","57,425.0"
160,25-3031,"Substitute Teachers, Short-Term",Bachelor's degree,"2,370.0","40,026.0","48,255.0","56,606.0"
148,25-2011,"Preschool Teachers, Except Special Education",Associate's degree,"1,420.0","40,477.0","52,133.0","64,306.0"
175,27-2022,Coaches and Scouts,Bachelor's degree,953.0,"43,418.0","67,276.0","96,418.0"
38,13-1151,Training and Development Specialists,Bachelor's degree,"1,458.0","49,090.0","82,571.0","111,130.0"
136,21-1021,"Child, Family, and School Social Workers",Bachelor's degree,"1,332.0","53,408.0","67,290.0","86,108.0"
40,13-1199,"Business Operations Specialists, All Other",Bachelor's degree,"2,024.0","55,732.0","91,532.0","123,004.0"
27,13-1011,"Agents and Business Managers of Artists, Perform...",Bachelor's degree,504.0,"55,969.0","97,374.0","194,662.0"
35,13-1121,"Meeting, Convention, and Event Planners",Bachelor's degree,828.0,"56,441.0","81,666.0","109,204.0"
145,23-2011,Paralegals and Legal Assistants,Associate's degree,"2,036.0","56,602.0","77,437.0","99,511.0"


entry ÷ median wage, degree-reachable jobs: median 0.69, openings-weighted 0.67 (a long climb where the ratio is low)


## 3. Pay over time: OEWS NY metro, real 2025 dollars
Metro, not NYC (twice the area). p25 stands in for early-career pay, p50 for the typical worker. 2019 is the base; 2025 drops top-coding and 2024 redrew the metro, so high-wage jobs across 2024 → 2025 carry a footnote (`series_note`).

In [5]:
oews = q("""
select year, soc_code, employment, annual_p25, annual_p50, annual_p25_2025usd, annual_p50_2025usd, quality_flag, series_note
from fct_job_employment_wage_annual where occ_level = 'detailed' and year in (2019, 2022, 2025)
""")
print(f"OEWS rows read (2019, 2022, 2025): {len(oews):,}")
O = oews.pivot_table(index="soc_code", columns="year", values=["annual_p25_2025usd", "annual_p50_2025usd", "annual_p25", "employment"])
O.columns = [f"{a}_{b}" for a, b in O.columns]
UO = U.merge(O, left_on="soc_code", right_index=True, how="inner")
UO["p25 real change %"] = (UO.annual_p25_2025usd_2025 / UO.annual_p25_2025usd_2019 - 1) * 100
UO["p50 real change %"] = (UO.annual_p50_2025usd_2025 / UO.annual_p50_2025usd_2019 - 1) * 100
UO["p25 nominal change %"] = (UO.annual_p25_2025 / UO.annual_p25_2019 - 1) * 100
w = UO.employment_2025
print(f"degree-reachable jobs with 2019 and 2025 metro pay: {int(UO['p25 real change %'].notna().sum())} | employment-weighted real change 2019 → 2025: "
      f"p25 {wavg(UO['p25 real change %'], w):+.1f}%, p50 {wavg(UO['p50 real change %'], w):+.1f}% (nominal p25 {wavg(UO['p25 nominal change %'], w):+.1f}%)")
allj = J[J.soc_level == "detailed"].merge(O, left_on="soc_code", right_index=True, how="inner")
allj["p25 real change %"] = (allj.annual_p25_2025usd_2025 / allj.annual_p25_2025usd_2019 - 1) * 100
allj["p50 real change %"] = (allj.annual_p50_2025usd_2025 / allj.annual_p50_2025usd_2019 - 1) * 100
print(f"all detailed jobs, same measure: p25 {wavg(allj['p25 real change %'], allj.employment_2025):+.1f}%, p50 {wavg(allj['p50 real change %'], allj.employment_2025):+.1f}%")
T = UO[["soc_code", "soc_title", "entry_education", "openings", "annual_p25_2025usd_2019", "annual_p25_2025usd_2025", "p25 real change %", "p50 real change %"]].rename(
    columns={"annual_p25_2025usd_2019": "p25 2019 (2025$)", "annual_p25_2025usd_2025": "p25 2025"})
big = T[(T.openings >= 500) & T["p25 real change %"].notna()]
show(big.nlargest(12, "p25 real change %").round(1), "Largest real gains in early-career pay (p25), jobs with 500+ openings")
show(big.nsmallest(12, "p25 real change %").round(1), "Largest real losses in early-career pay (p25), jobs with 500+ openings")
rows = []
for name, col in AREAS:
    d = UO[UO[col].fillna(False).astype(bool)]
    rows.append([name, len(d), wavg(d["p25 real change %"], d.employment_2025), wavg(d["p50 real change %"], d.employment_2025)])
show(pd.DataFrame(rows, columns=["area", "jobs", "p25 real change % (emp-weighted)", "p50 real change %"]).round(1), "By featured area")

OEWS rows read (2019, 2022, 2025): 2,318
degree-reachable jobs with 2019 and 2025 metro pay: 204 | employment-weighted real change 2019 → 2025: p25 +1.4%, p50 -1.2% (nominal p25 +25.9%)
all detailed jobs, same measure: p25 +7.2%, p50 +2.5%


**Largest real gains in early-career pay (p25), jobs with 500+ openings**

,soc_code,soc_title,entry_education,openings,p25 2019 (2025$),p25 2025,p25 real change %,p50 real change %
1343,53-2011,"Airline Pilots, Copilots, and Flight Engineers",Bachelor's degree,753.0,"112,386.3","185,910.0",65.4,98.6
494,27-3023,"News Analysts, Reporters, and Journalists",Bachelor's degree,505.0,"59,981.6","80,700.0",34.5,4.8
155,15-1255,Web and Digital Interface Designers,Bachelor's degree,601.7,"72,974.1","96,100.0",31.7,20.2
95,13-1082,Project Management Specialists,Bachelor's degree,"2,917.8","78,079.2","96,370.0",23.4,23.4
479,27-2022,Coaches and Scouts,Bachelor's degree,953.0,"39,847.0","49,140.0",23.3,10.7
92,13-1075,Labor Relations Specialists,Bachelor's degree,539.0,"69,533.5","83,760.0",20.5,8.7
824,41-3031,"Securities, Commodities, and Financial Services ...",Bachelor's degree,"4,145.0","85,258.6","100,830.0",18.3,7.9
406,25-2011,"Preschool Teachers, Except Special Education",Associate's degree,"1,420.0","36,654.7","42,920.0",17.1,3.8
152,15-1252,Software Developers,Bachelor's degree,"4,308.2","113,355.2","130,650.0",15.3,14.1
350,23-2011,Paralegals and Legal Assistants,Associate's degree,"2,036.0","53,348.7","61,040.0",14.4,7.5


**Largest real losses in early-career pay (p25), jobs with 500+ openings**

,soc_code,soc_title,entry_education,openings,p25 2019 (2025$),p25 2025,p25 real change %,p50 real change %
134,13-2099,"Financial Specialists, All Other",Bachelor's degree,563.1,"102,610.9","80,110.0",-21.9,-17.8
11,11-2011,Advertising and Promotions Managers,Bachelor's degree,725.0,"155,885.1","132,020.0",-15.3,-16.6
6,11-1021,General and Operations Managers,Bachelor's degree,"9,781.0","116,038.2","100,040.0",-13.8,-12.9
109,13-1199,"Business Operations Specialists, All Other",Bachelor's degree,"2,023.8","78,079.2","67,370.0",-13.7,-10.3
105,13-1151,Training and Development Specialists,Bachelor's degree,"1,458.0","65,918.9","56,920.0",-13.7,-12.4
174,17-1011,"Architects, Except Landscape and Naval",Bachelor's degree,749.0,"90,251.9","77,950.0",-13.6,-11.9
77,13-1011,"Agents and Business Managers of Artists, Perform...",Bachelor's degree,504.0,"67,223.1","58,610.0",-12.8,-12.0
168,15-2051,Data Scientists,Bachelor's degree,811.4,"112,796.2","101,320.0",-10.2,-7.8
13,11-2021,Marketing Managers,Bachelor's degree,"1,857.0","170,355.7","156,020.0",-8.4,-10.6
216,17-3011,Architectural and Civil Drafters,Associate's degree,539.0,"66,515.1","61,500.0",-7.5,0.8


**By featured area**

,area,jobs,p25 real change % (emp-weighted),p50 real change %
0,Tech,16,5.6,4.2
1,Healthcare,40,5.7,1.9
2,Business,50,-1.5,-3.4
3,Education,25,-0.1,-3.6


## 4. How hard is it to get hired as a newcomer: the D5 range
Three signals, none of which measures "entry-level openings" directly: BLS experience required (none / under 5 / 5+ years), ACS share of NYC workers in the job aged 22–29 (`high` ≥ 30%), O*NET share of incumbents who said they needed no prior related experience (`high` ≥ 25%). Bands were set in the step 4 plan before results.

In [6]:
sig = q("select soc_code, measure, value, band, moe_90, n, n_young, suppressed from fct_job_entry_signal")
print(f"fct_job_entry_signal rows: {len(sig):,}")
S = sig.pivot_table(index="soc_code", columns="measure", values="value", aggfunc="first")
Sb = sig.pivot_table(index="soc_code", columns="measure", values="band", aggfunc="first")
Sb.columns = [f"{c}_band" for c in Sb.columns]
US = U.merge(S, left_on="soc_code", right_index=True, how="left").merge(Sb, left_on="soc_code", right_index=True, how="left")
print(f"degree-reachable jobs with all three signals: {int(US[['acs_young_share', 'bls_experience', 'onet_newcomer']].notna().all(axis=1).sum())} of {len(US)}; "
      f"ACS suppressed (n < 30): {int(US.acs_young_share.isna().sum())}")
show(pd.crosstab(US.bls_experience_band, US.onet_newcomer_band, values=US.openings, aggfunc="sum").fillna(0).round(0),
     "Openings by BLS experience band × O*NET newcomer band")
show(pd.crosstab(US.bls_experience_band, US.acs_young_share_band, values=US.openings, aggfunc="sum").fillna(0).round(0),
     "Openings by BLS experience band × ACS young-worker band")
agree = US.dropna(subset=["acs_young_share_band", "onet_newcomer_band"])
print(f"ACS and O*NET bands agree on {int((agree.acs_young_share_band == agree.onet_newcomer_band).sum())} of {len(agree)} jobs "
      f"(Spearman of the two shares: {agree.acs_young_share.rank().corr(agree.onet_newcomer.rank()):.2f})")
open_all = US[(US.bls_experience_band == "none") & (US.onet_newcomer_band == "high") & (US.acs_young_share_band == "high")]
show(open_all.sort_values("openings", ascending=False)[["soc_code", "soc_title", "entry_education", "openings", "acs_young_share", "onet_newcomer"]].head(20).round(2),
     f"Open to newcomers on all three signals ({len(open_all)} jobs, {open_all.openings.sum():,.0f} openings a year)")
closed = US[(US.bls_experience_band != "none") | (US.onet_newcomer_band == "low")]
show(closed.sort_values("openings", ascending=False)[["soc_code", "soc_title", "entry_education", "openings", "bls_experience_band", "acs_young_share", "onet_newcomer"]].head(15).round(2),
     "Large jobs that want experience (BLS) or rarely take newcomers (O*NET low)")

fct_job_entry_signal rows: 2,252


degree-reachable jobs with all three signals: 124 of 221; ACS suppressed (n < 30): 85


**Openings by BLS experience band × O*NET newcomer band**

onet_newcomer_band,high,low,mid
bls_experience_band,,,
5_years_plus,0.0,"10,500.0","9,781.0"
none,"24,026.0","38,888.0","23,176.0"
under_5_years,57.0,"19,266.0","2,887.0"


**Openings by BLS experience band × ACS young-worker band**

...

ACS and O*NET bands agree on 45 of 124 jobs (Spearman of the two shares: 0.13)


**Open to newcomers on all three signals (6 jobs, 1,446 openings a year)**

,soc_code,soc_title,entry_education,openings,acs_young_share,onet_newcomer
168,15-2051,Data Scientists,Bachelor's degree,811.4,0.4,0.3
119,13-2041,Credit Analysts,Bachelor's degree,383.0,0.3,0.2
123,13-2053,Insurance Underwriters,Bachelor's degree,251.0,0.3,0.4
229,17-3029,"Engineering Technologists and Technicians, Excep...",Associate's degree,1.0,0.3,0.3
187,17-2041,Chemical Engineers,Bachelor's degree,0.0,0.6,0.3
305,19-4092,Forensic Science Technicians,Bachelor's degree,0.0,0.5,0.3


**Large jobs that want experience (BLS) or rarely take newcomers (O*NET low)**

,soc_code,soc_title,entry_education,openings,bls_experience_band,acs_young_share,onet_newcomer
6,11-1021,General and Operations Managers,Bachelor's degree,"9,781.0",5_years_plus,0.1,0.1
107,13-1161,Market Research Analysts and Marketing Specialists,Bachelor's degree,"6,037.0",none,0.4,0.0
97,13-1111,Management Analysts,Bachelor's degree,"4,444.0",under_5_years,0.3,0.0
152,15-1252,Software Developers,Bachelor's degree,"4,308.2",none,0.4,0.0
25,11-3031,Financial Managers,Bachelor's degree,"3,334.0",5_years_plus,0.1,0.1
90,13-1071,Human Resources Specialists,Bachelor's degree,"3,099.0",none,NaN,0.0
95,13-1082,Project Management Specialists,Bachelor's degree,"2,917.8",none,0.2,0.0
476,27-2012,Producers and Directors,Bachelor's degree,"2,542.0",under_5_years,0.2,0.1
496,27-3031,Public Relations Specialists,Bachelor's degree,"2,039.0",none,0.5,0.1
350,23-2011,Paralegals and Legal Assistants,Associate's degree,"2,036.0",none,0.3,0.1


### The hottest-jobs list through the D5 lens
The 25 degree-reachable jobs with the most openings (nb 40 List 1) with their entry wage and the three signals.

In [7]:
top25 = (U[~U.is_all_other].nlargest(25, "openings")[["soc_code", "soc_title", "entry_education", "openings"]]
         .merge(wage[["soc_code", "entry_wage"]], on="soc_code", how="left")
         .merge(US[["soc_code", "bls_experience_band", "acs_young_share", "onet_newcomer"]], on="soc_code", how="left"))
top25.index = range(1, 26)
show(top25.round(2), "List 1 jobs: pay and junior signals")

**List 1 jobs: pay and junior signals**

,soc_code,soc_title,entry_education,openings,entry_wage,bls_experience_band,acs_young_share,onet_newcomer
1,11-1021,General and Operations Managers,Bachelor's degree,"9,781.0","83,885.0",5_years_plus,0.1,0.1
2,29-1141,Registered Nurses,Bachelor's degree,"7,410.0","95,811.0",none,0.2,0.2
3,13-2011,Accountants and Auditors,Bachelor's degree,"6,648.0","78,988.0",none,0.2,0.4
4,13-1161,Market Research Analysts and Marketing Specialists,Bachelor's degree,"6,037.0","68,400.0",none,0.4,0.0
5,13-1111,Management Analysts,Bachelor's degree,"4,444.0","84,016.0",under_5_years,0.3,0.0
6,15-1252,Software Developers,Bachelor's degree,"4,308.2","121,283.0",none,0.4,0.0
7,41-3031,"Securities, Commodities, and Financial Services ...",Bachelor's degree,"4,145.0","97,955.0",none,0.3,0.1
8,25-2021,"Elementary School Teachers, Except Special Educa...",Bachelor's degree,"3,573.0","66,576.0",none,NaN,0.3
9,11-3031,Financial Managers,Bachelor's degree,"3,334.0","161,264.0",5_years_plus,0.1,0.1
10,13-1071,Human Resources Specialists,Bachelor's degree,"3,099.0","63,824.0",none,NaN,0.0


## 5. Skills, knowledge and software that matter most
O*NET importance (`IM`, 1–5) across the List 1 jobs, weighted by openings: what a graduate heading for any of these jobs will be asked for.

In [8]:
sk = q("""
select s.soc_code, s.domain, s.element_name, s.value
from fct_job_skill s where s.scale = 'IM' and s.suppress is not true
""")
print(f"fct_job_skill IM rows: {len(sk):,}")
top_codes = set(top25.soc_code)
SK = sk[sk.soc_code.isin(top_codes)].merge(top25[["soc_code", "openings"]], on="soc_code")
for dom, n in [("skill_essential", 10), ("skill_transferable", 12), ("knowledge", 12)]:
    t = SK[SK.domain == dom].groupby("element_name").apply(lambda d: wavg(d.value, d.openings), include_groups=False).sort_values(ascending=False).head(n)
    show(t.round(2).rename("importance (openings-weighted)").to_frame(), f"{dom}: top {n} across the 25 largest degree-reachable jobs")
rows = []
for name, col in AREAS:
    codes = set(U.loc[U[col].fillna(False).astype(bool), "soc_code"])
    d = sk[sk.soc_code.isin(codes) & (sk.domain == "knowledge")].merge(U[["soc_code", "openings"]], on="soc_code")
    t = d.groupby("element_name").apply(lambda x: wavg(x.value, x.openings), include_groups=False).nlargest(5)
    rows.append([name, ", ".join(f"{k} ({v:.1f})" for k, v in t.items())])
show(pd.DataFrame(rows, columns=["area", "top 5 knowledge areas (importance)"]), "Knowledge by featured area")

fct_job_skill IM rows: 53,654


**skill_essential: top 10 across the 25 largest degree-reachable jobs**

,importance (openings-weighted)
element_name,
Active Listening,3.9
Reading Comprehension,3.9
Critical Thinking,3.9
Speaking,3.9
Writing,3.7
Monitoring,3.6
Active Learning,3.5
Learning Strategies,3.1
Mathematics,2.8


**skill_transferable: top 12 across the 25 largest degree-reachable jobs**

,importance (openings-weighted)
element_name,
Judgment and Decision Making,3.6
Social Perceptiveness,3.6
Complex Problem Solving,3.5
Coordination,3.5
Time Management,3.3
Service Orientation,3.3
Persuasion,3.2
Instructing,3.1
Systems Analysis,3.1


**knowledge: top 12 across the 25 largest degree-reachable jobs**

,importance (openings-weighted)
element_name,
English Language,4.0
Customer and Personal Service,3.9
Administration and Management,3.5
Computers and Electronics,3.2
Mathematics,3.2
Administrative,3.1
Education and Training,3.0
Personnel and Human Resources,2.9
Communications and Media,2.8


**Knowledge by featured area**

,area,top 5 knowledge areas (importance)
0,Tech,"Computers and Electronics (4.6), English Languag..."
1,Healthcare,"Customer and Personal Service (4.3), English Lan..."
2,Business,"English Language (3.9), Customer and Personal Se..."
3,Education,"English Language (4.4), Education and Training (..."


In [9]:
sw = q("select soc_code, product, is_hot_technology, is_in_demand from fct_job_software")
print(f"fct_job_software rows: {len(sw):,}")
SW = sw[sw.soc_code.isin(set(U.soc_code))].merge(U[["soc_code", "openings"]], on="soc_code")
hot = SW[SW.is_hot_technology].groupby("product").agg(jobs=("soc_code", "nunique"), openings=("openings", "sum")).sort_values("openings", ascending=False).head(20)
show(hot.round(0), "Hot technologies listed for the most degree-reachable openings")
for name, col in AREAS:
    codes = set(U.loc[U[col].fillna(False).astype(bool), "soc_code"])
    t = SW[SW.soc_code.isin(codes) & SW.is_hot_technology].groupby("product").openings.sum().nlargest(8)
    md(f"**{name}** hot technologies by openings: " + ", ".join(t.index))

fct_job_software rows: 29,433


**Hot technologies listed for the most degree-reachable openings**

,jobs,openings
product,,
Microsoft Office software,210,"131,730.0"
Microsoft Excel,209,"131,467.0"
Microsoft Word,205,"128,121.0"
Microsoft PowerPoint,201,"126,904.0"
Microsoft Outlook,180,"124,088.0"
Microsoft Access,152,"103,547.0"
Microsoft SharePoint,84,"100,994.0"
Microsoft Project,106,"96,119.0"
SAP software,105,"86,737.0"


**Tech** hot technologies by openings: Linux, Microsoft Access, Microsoft Excel, Microsoft Office software, Microsoft Visual Basic, Oracle Java, Perl, Python

**Healthcare** hot technologies by openings: Microsoft Excel, Microsoft Office software, Microsoft Word, Microsoft Outlook, Microsoft PowerPoint, Microsoft Access, MEDITECH software, eClinicalWorks EHR software

**Business** hot technologies by openings: Microsoft Excel, Microsoft Office software, Microsoft PowerPoint, Microsoft Word, Microsoft Outlook, Microsoft Access, Microsoft SharePoint, SAP software

**Education** hot technologies by openings: Microsoft Excel, Microsoft Office software, Microsoft Word, Microsoft PowerPoint, Microsoft Outlook, Microsoft SharePoint, Google Docs, Microsoft Access

## 6. Checks

In [10]:
chk = []
s = att.groupby("soc_code").pct_of_workers.sum()
chk.append(["attainment shares sum to 100 per job (max |gap|)", f"{(s - 100).abs().max():.2f} over {len(s)} jobs"])
m = UW.merge(O.annual_p25_2025.rename("metro_p25_2025"), left_on="soc_code", right_index=True, how="inner")
chk.append(["NYC entry wage vs metro p25 2025 (independent surveys): Pearson r, median ratio",
            f"{m.entry_wage.corr(m.metro_p25_2025):.2f}, {(m.entry_wage / m.metro_p25_2025).median():.2f}"])
sd = US[US.soc_code == "15-1252"].iloc[0]
chk.append(["Trace Software Developers 15-1252 (step 4 §9: young share 0.369 high, newcomer 0.044 low, BLS none, 51.7% bachelor's)",
            f"young {sd.acs_young_share:.3f} {sd.acs_young_share_band}, newcomer {sd.onet_newcomer:.3f} {sd.onet_newcomer_band}, BLS {sd.bls_experience_band}, "
            f"bachelor's {A.loc['15-1252', 'bachelor\'s']:.1f}%, NYC entry ${wage.set_index('soc_code').loc['15-1252', 'entry_wage']:,.0f}, hot tech {int(sw[(sw.soc_code == '15-1252') & sw.is_hot_technology]['product'].nunique())}"])
cpi = q("select year, area_key, deflator_to_2025 from fct_cpi_annual where area_key = 'ny_metro' and year in (2019, 2025)").set_index("year").deflator_to_2025
chk.append(["CPI NY metro deflator 2019 → 2025 (plan: +24.2%)", f"{(cpi[2019] - 1) * 100:+.1f}%, 2025 = {cpi[2025]:.3f}"])
med = sig[sig.measure == "onet_newcomer"].merge(sig[sig.measure == "bls_experience"][["soc_code", "band"]].rename(columns={"band": "bls"}), on="soc_code")
chk.append(["D5 direction: median O*NET newcomer share by BLS band (none / under 5 / 5+)",
            " / ".join(f"{med[med.bls == b].value.median():.3f}" for b in ["none", "under_5_years", "5_years_plus"])])
show(pd.DataFrame(chk, columns=["check", "result"]), "Checks")

**Checks**

,check,result
0,attainment shares sum to 100 per job (max |gap|),0.20 over 832 jobs
1,NYC entry wage vs metro p25 2025 (independent su...,"0.95, 0.93"
2,Trace Software Developers 15-1252 (step 4 §9: yo...,"young 0.369 high, newcomer 0.044 low, BLS none, ..."
3,CPI NY metro deflator 2019 → 2025 (plan: +24.2%),"+24.2%, 2025 = 1.000"
4,D5 direction: median O*NET newcomer share by BLS...,0.228 / 0.115 / 0.000


## 7. Findings (from the tables above)

1. **What it takes.** 93% of degree-reachable openings (124,876 of 133,684 a year) are in bachelor's-entry jobs; associate's-entry jobs supply 6.6%. 68% of openings need no prior related experience, 17% under five years, 15% five years or more, and that last group is the managers at the top of List 1 (General and Operations Managers 9,781 openings, Financial, Marketing, Sales and IS Managers). 88% need no on-the-job training; 87% are O*NET job zone 4.
2. The entry label and the workforce differ. In associate's-entry jobs 45% of workers hold a bachelor's or more (Paralegals 50%, Preschool Teachers 54%). In bachelor's-entry jobs 28% of workers have less than a bachelor's; for General and Operations Managers it is 53%, Construction Managers 65%, Postsecondary Teaching Assistants 66%. 113 of the 221 jobs carry a profile BLS borrowed from a broader Census code, so these shares are indicative.
3. **Starting pay in NYC.** Entry wage (lower third of all workers, not new graduates) for the median degree-reachable job is $69,556; the spread runs $49 k (p10) to $115 k (p90). Openings-weighted: $59,876 in associate's-entry jobs, $84,747 in bachelor's-entry jobs. By featured area: Tech $110,761, Business $88,720, Healthcare $83,098, Education $62,481. The large low-entry jobs are in education and social service (Postsecondary Teaching Assistants $37,904, Substitute Teachers $40,026, Preschool Teachers $40,477, Child and Family Social Workers $53,408). Entry pay is two thirds of the median wage (0.67 openings-weighted), so the climb is long everywhere. None of the 22 wage-floor jobs is degree-reachable.
4. Real pay for degree jobs has not moved since 2019. NY metro, employment-weighted, 2019 → 2025 in 2025 dollars: p25 +1.4%, p50 −1.2% for degree-reachable jobs, against +7.2% / +2.5% for all jobs (nominal p25 +25.9% ≈ CPI +24.2%). Lower-wage jobs gained in real terms, degree jobs did not. Business −1.5% (p25) / −3.4% (p50), Education −0.1% / −3.6%, Tech +5.6% / +4.2%, Healthcare +5.7% / +1.9%. Managers lost the most (General and Operations Managers p25 −13.8%, Advertising Managers −15.3%, Marketing Managers −8.4%); Project Management Specialists +23%, Software Developers +15%, Paralegals +14%, Preschool Teachers +17% gained. 2025 drops top-coding, so high-wage p50 moves across 2024 → 2025 carry a footnote.
5. The three junior signals do not agree, which is why D5 is a range. ACS young-worker share and O*NET newcomer share rank jobs almost independently (Spearman 0.13; same band on 45 of 124 jobs). BLS says 68% of openings need no experience, but O*NET incumbents report a low newcomer share for 69 k of those openings (Market Research Analysts 0.00, Software Developers 0.04, Management Analysts 0.00, HR Specialists 0.00): "no experience required" is not "hires beginners". Only 6 jobs are open on all three signals (Data Scientists, Credit Analysts, Insurance Underwriters and three tiny ones). 85 of 221 jobs have the ACS signal suppressed (n < 30). The D5 direction check holds (median newcomer share 0.228 / 0.115 / 0.000 across the BLS bands).
6. What the big jobs ask for. Across the 25 largest degree-reachable jobs the top essentials are Active Listening, Reading Comprehension, Critical Thinking and Speaking (3.9 of 5), then Writing (3.7); transferable: Judgment and Decision Making, Social Perceptiveness (3.6), Complex Problem Solving, Coordination (3.5); knowledge: English (4.0), Customer and Personal Service (3.9), Administration and Management (3.5), Computers and Electronics and Mathematics (3.2). Software: Microsoft Office tools head every list; SAP, SQL, Salesforce, SAS, SPSS and PeopleSoft are the named systems with the most openings behind them. Tech's top knowledge is Computers and Electronics (4.6); Healthcare's and Education's is Customer Service and English, not science.
7. **Checks.** Attainment shares sum to 100 ± 0.2 on all 832 jobs; NYC entry wage vs metro p25 (independent surveys) r = 0.95, median ratio 0.93; Software Developers trace matches step 4 §9 (young 0.369 high, newcomer 0.044 low, BLS none, 51.7% bachelor's); CPI +24.2%; D5 direction reproduced.
8. **Limits.** Entry wage is a market floor for all workers, not a graduate salary; pay over time is metro, not NYC; half the attainment profiles are borrowed; O*NET newcomer share is incumbents' self-report; ACS shares are suppressed under 30 responses.
9. **For the mart.** Pay page: NYS DOL entry / median / experienced for NYC plus the metro real p25 trend as the "since 2019" line, with the top-code footnote. Junior-friendliness as three labeled signals side by side, never a blended score (D5, O1); an "asks for 5+ years" flag on the 15% of openings that do. Skills page: the shared top-10 across List 1 plus the area-specific knowledge, and the named software beyond Office.